# ポケモンのタイプはステータスに影響するのか？
### 学籍番号:1264810130
### 氏名:神郡正稀 

問いは、「タイプごとにステータスの特徴があるのではないか」です。<br>
ポケモンにはタイプがあって、すべてのポケモンはなにかしらのタイプに分類されています。<br>
みんなも、いわタイプのぼうぎょは高い、かくとうタイプのこうげきは高いみたいなイメージはあると思います。<br>
それを実際にpythonで調べてみました。

## 1.ポケモンのデータを読む

まずは、Pokemon.csvを使って、dfにします。Pokemon.csvでは最新のポケモンまでの図鑑No、名前、タイプ、種族値（ステータス）などがまとめられています。

In [17]:
import pandas as pd
df = pd.read_csv('Pokemon.csv')
print(df.shape)

(1215, 13)


このPokemon.csvでは見た目が違うだけの同じポケモンが何匹か含まれているので、それを除いていきます。

In [18]:
P_ID = df.drop_duplicates('ID') #P_IDは図鑑No.。
print(len(P_ID))

1025


現在のポケモンの総数は1025匹なので、正しく整理出来ました。

## 2.タイプと種族値の辞書を作る

次に、後で確認しやすいように辞書を作ります。<br>
Pokemon.csvはタイプ名や種族値の名前が英語表記なので、分かりやすいように日本語にします。

In [19]:
type_names = {
    'ノーマル':'Normal','くさ':'Grass','ほのお':'Fire','みず':'Water',
    'でんき':'Electric','むし':'Bug','ひこう':'Flying','いわ':'Rock',
    'どく':'Poison','じめん':'Ground','こおり':'Ice','かくとう':'Fighting',
    'エスパー':'Psychic','ゴースト':'Ghost','ドラゴン':'Dragon',
    'あく':'Dark','はがね':'Steel','フェアリー':'Fairy'
}

stats_names = {
    'HP':'HP','こうげき':'Attack','ぼうぎょ':'Defense',
    'とくこう':'Sp. Atk','とくぼう':'Sp. Def','すばやさ':'Speed'
}

## 3.全タイプの平均種族値を集計する

いよいよ種族値を集計します。一旦表示はせずに集計のみ。

In [20]:
results = {}

for type_jp, t in type_names.items():
    total = {jp: 0 for jp in stats_names}
    count = 0
    for index, row in P_ID.iterrows():
        if row['Type1'] == t or row['Type2'] == t:
            count += 1
            for jp, col in stats_names.items():
                total[jp] += row[col]

    results[type_jp] = {}
    for jp in stats_names:
        results[type_jp][jp] = total[jp] / count

## 4.結果を表示する

集められたので、printで表示します。<br>
タイプごとに行が分かれるようにしました。

In [21]:
for type_jp in type_names:
    print(type_jp, end='  ')
    for jp in stats_names:
        print(f'{results[type_jp][jp]:.1f}', end='  ')
    print()

ノーマル  76.6  74.0  60.3  59.1  63.9  68.9  
くさ  66.1  74.8  71.7  71.7  71.8  59.7  
ほのお  69.9  80.6  67.5  86.3  71.0  74.1  
みず  69.6  72.1  72.2  71.0  68.6  65.6  
でんき  68.7  74.9  64.9  84.2  67.7  81.9  
むし  56.6  68.0  69.2  56.8  64.2  60.2  
ひこう  70.4  76.7  65.6  71.6  68.5  82.4  
いわ  68.9  87.2  103.5  58.5  72.3  51.8  
どく  66.0  71.6  66.3  71.0  69.4  68.5  
じめん  77.1  88.4  85.5  57.5  64.8  58.5  
こおり  80.1  82.3  75.7  74.9  75.2  65.4  
かくとう  77.8  104.3  76.4  63.6  69.7  75.4  
エスパー  73.8  67.3  72.6  88.0  84.2  71.6  
ゴースト  65.7  77.4  77.5  81.6  79.1  66.2  
ドラゴン  84.1  94.0  83.0  87.2  78.3  78.2  
あく  76.3  93.6  71.4  72.7  68.9  76.8  
はがね  71.0  90.7  107.9  71.1  78.9  60.2  
フェアリー  67.2  64.7  70.1  77.2  83.9  65.7  


## 5.表にして整理

tableを使って表にして見やすくします。

In [22]:
table = pd.DataFrame(results).T.round(1) #roundで小数をまとめる
table

,HP,こうげき,ぼうぎょ,とくこう,とくぼう,すばやさ
ノーマル,76.6,74.0,60.3,59.1,63.9,68.9
くさ,66.1,74.8,71.7,71.7,71.8,59.7
ほのお,69.9,80.6,67.5,86.3,71.0,74.1
みず,69.6,72.1,72.2,71.0,68.6,65.6
でんき,68.7,74.9,64.9,84.2,67.7,81.9
むし,56.6,68.0,69.2,56.8,64.2,60.2
ひこう,70.4,76.7,65.6,71.6,68.5,82.4
いわ,68.9,87.2,103.5,58.5,72.3,51.8
どく,66.0,71.6,66.3,71.0,69.4,68.5
じめん,77.1,88.4,85.5,57.5,64.8,58.5


## 6.全体平均との差を表示

すべてのタイプの平均とそれぞれのタイプの平均の差を表示しました。<br>
正の数は平均よりも高く、負の数は平均よりも低いことを表しています。

In [23]:
overall = P_ID[list(stats_names.values())].mean()   # 6つの全体平均種族値
overall.index = list(stats_names.keys())

diff = (table - overall).round(1)
diff

,HP,こうげき,ぼうぎょ,とくこう,とくぼう,すばやさ
ノーマル,6.4,-3.5,-12.2,-11.0,-6.3,1.7
くさ,-4.1,-2.7,-0.8,1.6,1.6,-7.5
ほのお,-0.3,3.1,-5.0,16.2,0.8,6.9
みず,-0.6,-5.4,-0.3,0.9,-1.6,-1.6
でんき,-1.5,-2.6,-7.6,14.1,-2.5,14.7
むし,-13.6,-9.5,-3.3,-13.3,-6.0,-7.0
ひこう,0.2,-0.8,-6.9,1.5,-1.7,15.2
いわ,-1.3,9.7,31.0,-11.6,2.1,-15.4
どく,-4.2,-5.9,-6.2,0.9,-0.8,1.3
じめん,6.9,10.9,13.0,-12.6,-5.4,-8.7


## 7.自由に表の並び替えを出来るようにする

ソート機能が欲しいと思ったので、if文とinputでそれっぽく作ってみました。<br>
注意点として、全角にする必要があります。（日本語のみ）<br>
もしかしたら拡張機能で使いやすいのがあったかも。

In [ ]:
table_column = input('並べ替える列(HP/こうげき/ぼうぎょ/とくこう/とくぼう/すばやさ)を打ってください: ')
table_row = input('高い順なら 高い、低い順なら 低いと打ってください（ひらがなでもOK）: ')

if table_column in table.columns:
    if table_row == '高い' or 'たかい':
        display(table.sort_values(table_column, ascending=False))
        print(f'{table_column}の高い順で表示中')
    else :
        display(table.sort_values(table_column, ascending=True))
        print(f'{table_column}の低い順で表示中')
else:
    print('列の名前が正しくありません')

,HP,こうげき,ぼうぎょ,とくこう,とくぼう,すばやさ
かくとう,77.8,104.3,76.4,63.6,69.7,75.4
ドラゴン,84.1,94.0,83.0,87.2,78.3,78.2
あく,76.3,93.6,71.4,72.7,68.9,76.8
はがね,71.0,90.7,107.9,71.1,78.9,60.2
じめん,77.1,88.4,85.5,57.5,64.8,58.5
いわ,68.9,87.2,103.5,58.5,72.3,51.8
こおり,80.1,82.3,75.7,74.9,75.2,65.4
ほのお,69.9,80.6,67.5,86.3,71.0,74.1
ゴースト,65.7,77.4,77.5,81.6,79.1,66.2
ひこう,70.4,76.7,65.6,71.6,68.5,82.4


こうげきの順で表示中


こうげき　や　ぼうぎょ　は全角で打つ必要があるのに対し、

## 8.分かったこと

いわタイプのぼうぎょは平均よりも31.0高く、かくとうタイプのこうげきは平均よりも26.8高いことから、<br>予想通りポケモンのタイプごとの種族値の特徴はあることが分かりました。<br>

また、ドラゴンタイプは全体的に平均よりも高く、逆にむしタイプは全体的に平均よりも低いことが分かりました。<br>
いわタイプ、じめんタイプ、はがねタイプのぼうぎょが高かったり、でんきタイプやひこうタイプのすばやさが高かったり、全体的に結構イメージ通りかも。<br>

意外だったのはノーマルタイプの種族値は平均よりも少し離れていること。ノーマルというからにはもっと平均に寄っているのかと思っていた。<br>

終わってから気が付いたのですが、メガシンカなど一部のポケモンを集計に入れられなかった可能性があります。完璧な表ではないかもしれない。<br>

以上です。楽しかった。<br>
csvの出典: https://github.com/lgreski/pokemonData
